In [13]:
import pandas as pd
import re
from stemmid import Stemmer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# KONFIGURASI
INPUT_FILE = "../data/dataset_TA.xlsx"
OUTPUT_FILE = "../data/dataset_preprocessed.csv"
RANDOM_STATE = 42

# PETA EMOJI → TOKEN SENTIMEN
EMOJI_MAP = {
    "😡": " emosi_negatif ", "😠": " emosi_negatif ", "😤": " emosi_negatif ",
    "😞": " emosi_negatif ", "😢": " emosi_negatif ", "😭": " emosi_negatif ",
    "👎": " emosi_negatif ", "💔": " emosi_negatif ",
    "🙂": " emosi_positif ", "😊": " emosi_positif ", "😄": " emosi_positif ",
    "😁": " emosi_positif ", "😍": " emosi_positif ", "👍": " emosi_positif ",
    "❤️": " emosi_positif "
}

# =========================
# STOPWORD (dari Sastrawi)
# =========================
factory = StopWordRemoverFactory()
STOPWORDS = set(factory.get_stop_words())

# JAGA kata penting untuk sentimen
STOPWORDS -= {"tidak", "kurang", "belum"}

# =========================
# LOAD DATASET
# =========================
print("Membaca dataset...")
df_raw = pd.read_excel(INPUT_FILE)
df = df_raw.copy()
df['Label'] = df['Label'].str.lower()

print("\nDistribusi Data Natural:")
print(df['Label'].value_counts())

# =========================
# STEMMER
# =========================
print("\nMenyiapkan Stemmer stemmid...")
stemmer = Stemmer()

# =========================
# FUNGSI PREPROCESSING
# =========================
def normalisasi_emoji(text):
    for emoji, token in EMOJI_MAP.items():
        text = text.replace(emoji, token)
    return text

def normalisasi_teks(text):
    text = str(text).lower()
    text = normalisasi_emoji(text)

    # cleaning
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    # tokenizing
    tokens = text.split()

    # stopword removal
    tokens = [t for t in tokens if t not in STOPWORDS]

    # gabung lagi
    text = " ".join(tokens)

    # stemming
    text = stemmer.loads(text)

    return text

# =========================
# PROSES NLP
# =========================
print("Memulai proses pembersihan teks...")
df['text_processed'] = df['Ulasan'].apply(normalisasi_teks)

# hapus kosong
df = df[df['text_processed'].str.strip() != ""]

# =========================
# SHUFFLE & SIMPAN
# =========================
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSelesai! Dataset tersimpan sebagai '{OUTPUT_FILE}'")

Membaca dataset...

Distribusi Data Natural:
Label
positif    156
negatif     45
Name: count, dtype: int64

Menyiapkan Stemmer stemmid...
Memulai proses pembersihan teks...

Selesai! Dataset tersimpan sebagai '../data/dataset_preprocessed.csv'


In [14]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB          # sebelumnya ComplementNB
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import RandomOverSampler

# KONFIGURASI
INPUT_FILE = "../data/dataset_preprocessed.csv"
MODEL_FILE = "../data/model_mnb.pkl"         # sebelumnya model_cnb.pkl
VECTORIZER_FILE = "../data/vectorizer.pkl"
TEST_DATA_FILE = "../data/data_ujian.csv"
RANDOM_STATE = 42

# 1. LOAD DATA
df = pd.read_csv(INPUT_FILE).dropna(subset=['text_processed', 'Label'])
X = df['text_processed']
y = df['Label']

# 2. STRATIFIED SPLIT (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# 3. TF-IDF VECTORIZATION
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 4. OVERSAMPLING DENGAN RandomOverSampler   ← BLOK BARU, TIDAK ADA SEBELUMNYA
ros = RandomOverSampler(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = ros.fit_resample(X_train_vec, y_train)
print("Distribusi setelah oversampling:", pd.Series(y_train_bal).value_counts().to_dict())

# 5. TRAINING MODEL
model = MultinomialNB(alpha=1.0)               # ← GANTI: sebelumnya ComplementNB
model.fit(X_train_bal, y_train_bal)            # ← GANTI: sebelumnya X_train_vec, y_train

print("Urutan Kelas di Model:", model.classes_)

# 6. SIMPAN MODEL & DATA UJI
joblib.dump(model, MODEL_FILE)
joblib.dump(vectorizer, VECTORIZER_FILE)

df_test = pd.DataFrame({
    "text_processed": X_test,
    "Label": y_test
})
df_test.to_csv(TEST_DATA_FILE, index=False)

print("=== PROSES SELESAI ===")
print(f"Model berhasil dilatih dengan {X_train_bal.shape[0]} data (setelah oversampling).")
print(f"Akurasi pada data uji: {accuracy_score(y_test, model.predict(X_test_vec))*100:.2f}%")
print(f"Data uji disimpan di: {TEST_DATA_FILE}")

Distribusi setelah oversampling: {'positif': 124, 'negatif': 124}
Urutan Kelas di Model: ['negatif' 'positif']
=== PROSES SELESAI ===
Model berhasil dilatih dengan 248 data (setelah oversampling).
Akurasi pada data uji: 82.93%
Data uji disimpan di: ../data/data_ujian.csv


In [15]:
import pandas as pd
import joblib
import re
import numpy as np
from stemmid import Stemmer

# 1. LOAD MODEL
model = joblib.load("../data/model_mnb.pkl")
vectorizer = joblib.load("../data/vectorizer.pkl")

# 2. SIMULASI KAMUS KUSTOM (Input dari Pegawai)
# Di aplikasi web nanti, ini diambil dari database
KAMUS_KUSTOM = {
    "kacau": "negatif",
    "lelet": "negatif",
    "parah": "negatif",
    "nyaman": "positif"
}

# 3. FUNGSI XAI (Penjelasan Skor per Kata)
def get_xai_explanation(text, model, vectorizer):
    feature_names = vectorizer.get_feature_names_out()
    explanation = []
    classes = model.classes_
    idx_neg = np.where(classes == 'negatif')[0][0]
    idx_pos = np.where(classes == 'positif')[0][0]
    
    # KUNCI PERBAIKAN: Gunakan regex \b (word boundary) 
    # Agar 'si' tidak terdeteksi di dalam 'bersih'
    for feature in feature_names:
        # Mencari fitur sebagai kata utuh di dalam teks
        if re.search(r'\b' + re.escape(feature) + r'\b', text):
            idx_f = np.where(feature_names == feature)[0][0]
            
            w_neg = model.feature_log_prob_[idx_neg][idx_f]
            w_pos = model.feature_log_prob_[idx_pos][idx_f]
            
            if w_neg < w_pos:
                sentimen = "Positif"
                kekuatan = round(w_pos - w_neg, 2)
            else:
                sentimen = "Negatif"
                kekuatan = round(w_neg - w_pos, 2)
                
            explanation.append({"fitur": feature, "arah": sentimen, "kekuatan": kekuatan})
            
    # Urutkan berdasarkan kekuatan tertinggi
    explanation = sorted(explanation, key=lambda x: x['kekuatan'], reverse=True)
    return explanation

# 4. LOGIKA PREDIKSI HYBRID (CNB + KAMUS KUSTOM)
def hybrid_prediction(text, model, vectorizer, custom_dict):
    # a. Bersihkan & Transform
    teks_bersih = stemmer.loads(text.lower())
    vec = vectorizer.transform([teks_bersih])
    
    # b. Prediksi Dasar (CNB)
    prob = model.predict_proba(vec)[0]
    label_cnb = model.predict(vec)[0]
    conf_cnb = max(prob) * 100
    
    # c. Cek Kamus Kustom (Boosting)
    boost_label = None
    for word in teks_bersih.split():
        if word in custom_dict:
            boost_label = custom_dict[word]
            break # Jika ada kata kustom, kita beri perhatian lebih
    
    # Jika ada di kamus kustom, kita naikkan keyakinannya atau timpa labelnya
    final_label = boost_label if boost_label else label_cnb
    final_conf = 99.9 if boost_label else conf_cnb
    
    return final_label, final_conf, teks_bersih

# --- DEMO DASHBOARD ---
stemmer = Stemmer()

while True:
    ulasan = input("\nMasukkan ulasan: ")
    if ulasan.lower() == 'exit': break

    label, conf, bersih = hybrid_prediction(ulasan, model, vectorizer, KAMUS_KUSTOM)
    expl = get_xai_explanation(bersih, model, vectorizer)

    print(f"\n[ HASIL DASHBOARD ]")
    print(f"> Sentimen : {label.upper()}")
    print(f"> Keyakinan: {conf:.2f}%")
    
    print(f"\n[ ANALISIS XAI (SKOR FITUR) ]")
    for item in expl:
    # Gunakan item['fitur'] karena sekarang bisa berisi 2 kata (bigram)
        print(f"- Fitur '{item['fitur']}' cenderung {item['arah']} (Kekuatan: {item['kekuatan']})")


[ HASIL DASHBOARD ]
> Sentimen : NEGATIF
> Keyakinan: 58.25%

[ ANALISIS XAI (SKOR FITUR) ]
- Fitur 'ga' cenderung Negatif (Kekuatan: 1.47)
- Fitur 'beda' cenderung Positif (Kekuatan: 0.38)
- Fitur 'belum' cenderung Negatif (Kekuatan: 0.25)
- Fitur 'ada' cenderung Positif (Kekuatan: 0.18)
- Fitur 'cuci' cenderung Positif (Kekuatan: 0.1)

[ HASIL DASHBOARD ]
> Sentimen : NEGATIF
> Keyakinan: 85.09%

[ ANALISIS XAI (SKOR FITUR) ]
- Fitur 'kurang' cenderung Negatif (Kekuatan: 1.57)
- Fitur 'kurang bersih' cenderung Negatif (Kekuatan: 1.34)
- Fitur 'bersih' cenderung Positif (Kekuatan: 0.46)

[ HASIL DASHBOARD ]
> Sentimen : NEGATIF
> Keyakinan: 74.98%

[ ANALISIS XAI (SKOR FITUR) ]
- Fitur 'tidak' cenderung Negatif (Kekuatan: 1.03)
- Fitur 'tidak bersih' cenderung Negatif (Kekuatan: 0.82)
- Fitur 'bersih' cenderung Positif (Kekuatan: 0.46)

[ HASIL DASHBOARD ]
> Sentimen : POSITIF
> Keyakinan: 81.03%

[ ANALISIS XAI (SKOR FITUR) ]
- Fitur 'mantap' cenderung Positif (Kekuatan: 1.45)
